# Predicting Product Demand in a Retail Store


## Introduction
The aim of this project is to predict the demand for products in a retail store chain. By leveraging historical sales data, pricing, promotions, and other relevant factors, we can forecast future demand and optimize inventory management.


## Task 1: Data Exploration and Preprocessing

In [ ]:

# Import necessary libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# For preprocessing
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer

# Load the dataset
df = pd.read_csv('/mnt/data/product_demand_prediction_dataset.csv')  # Modify the path as necessary

# Display first few rows
print(df.head())

# Summary statistics
print(df.describe())

# Data types and non-null counts
print(df.info())


### 1.3 Handling Missing Values

In [ ]:

# Check for missing values
print(df.isnull().sum())

# Visualize missing values
sns.heatmap(df.isnull(), cbar=False, cmap='viridis')
plt.title('Missing Values Heatmap')
plt.show()

# Handle missing values (imputation example)
numerical_cols = ['Sales', 'Price', 'CompetitorPrice', 'EconomicIndicator', 'StockLevel', 'Demand']
categorical_cols = ['ProductID', 'StoreID', 'Promotion', 'Season', 'Holiday', 'DayOfWeek', 'Weather']

# Impute numerical columns with median
num_imputer = SimpleImputer(strategy='median')
df[numerical_cols] = num_imputer.fit_transform(df[numerical_cols])

# Impute categorical columns with mode
cat_imputer = SimpleImputer(strategy='most_frequent')
df[categorical_cols] = cat_imputer.fit_transform(df[categorical_cols])

# Verify no missing values remain
print(df.isnull().sum())


### 1.4 Encoding Categorical Variables

In [ ]:

# Convert categorical variables into numerical format
# Initialize OneHotEncoder
ohe = OneHotEncoder(drop='first', sparse=False)

# Columns to encode
ohe_cols = ['Promotion', 'Season', 'Holiday', 'DayOfWeek', 'Weather']

# Perform One-Hot Encoding
encoded_ohe = pd.DataFrame(ohe.fit_transform(df[ohe_cols]), columns=ohe.get_feature_names_out(ohe_cols))

# Concatenate with the original dataframe
df = pd.concat([df.drop(ohe_cols, axis=1), encoded_ohe], axis=1)

# Label Encoding for 'ProductID' and 'StoreID'
from sklearn.preprocessing import LabelEncoder
le = LabelEncoder()
df['ProductID'] = le.fit_transform(df['ProductID'])
df['StoreID'] = le.fit_transform(df['StoreID'])

print(df.head())


## Task 2: Feature Engineering

In [ ]:

# Feature Scaling
features_to_scale = ['Sales', 'Price', 'CompetitorPrice', 'EconomicIndicator', 'StockLevel']
scaler = StandardScaler()
df[features_to_scale] = scaler.fit_transform(df[features_to_scale])

# Creating new features (PriceChange and Moving Average Sales)
df['PriceChange'] = df['Price'] - df['CompetitorPrice']
df['Date'] = pd.to_datetime(df['Date'])  # Ensure the Date column is in datetime format
df = df.sort_values(by=['ProductID', 'Date'])
df['Sales_MA7'] = df.groupby('ProductID')['Sales'].transform(lambda x: x.rolling(window=7, min_periods=1).mean())

# Display changes
print(df.head())


## Task 3: Model Building

In [ ]:

# Splitting the dataset into training and testing sets
from sklearn.model_selection import train_test_split

X = df.drop(['Date', 'Demand'], axis=1)  # Feature matrix
y = df['Demand']  # Target variable

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f'Training set size: {X_train.shape}')
print(f'Testing set size: {X_test.shape}')

# Train models
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor

# Initialize models
models = {
    'Linear Regression': LinearRegression(),
    'Decision Tree': DecisionTreeRegressor(random_state=42),
    'Random Forest': RandomForestRegressor(random_state=42),
    'Gradient Boosting': GradientBoostingRegressor(random_state=42)
}

# Train models
for name, model in models.items():
    model.fit(X_train, y_train)
    print(f'{name} trained.')


## Task 4: Model Evaluation

In [ ]:

# Evaluation using MAE, MSE, and R-squared
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np

def evaluate_model(model, X_test, y_test):
    predictions = model.predict(X_test)
    mae = mean_absolute_error(y_test, predictions)
    mse = mean_squared_error(y_test, predictions)
    rmse = np.sqrt(mse)
    r2 = r2_score(y_test, predictions)
    return mae, mse, rmse, r2

# Evaluate all trained models
evaluation_results = {}
for name, model in models.items():
    mae, mse, rmse, r2 = evaluate_model(model, X_test, y_test)
    evaluation_results[name] = {'MAE': mae, 'MSE': mse, 'RMSE': rmse, 'R2': r2}

# Display results
eval_df = pd.DataFrame(evaluation_results).T
print(eval_df)


## Task 5: Insights and Recommendations


### Insights:
- Pricing, promotions, and seasonal factors significantly impact product demand.
- Economic conditions and competitor pricing are also crucial in determining product demand.

### Recommendations:
- Implement dynamic pricing strategies to optimize sales and reduce overstock situations.
- Schedule promotions strategically based on demand forecasts and seasonality.
- Monitor competitor prices and adjust inventory to stay competitive.


## Conclusion


By building a machine learning model to predict product demand, we can provide valuable insights to the retail store chain. These insights help optimize inventory management, reducing both stockouts and overstock situations. The models trained in this project provide a foundation for future improvements and refinements, such as incorporating more advanced feature engineering and ensemble methods.
